# 03 — Export and Validation

Build an `EModelExportAndValidationScanConfig`, expand it via `GridScanGenerationTask`,
and run the `EModelExportAndValidationTask` for each coordinate. The task downloads
the optimisation `TaskResult` assets (checkpoints, `final.json`, recipes, params,
figures, hoc, sonata) from entitycore, seeds its working directory, runs
`pipeline.store_optimisation_results()` → `pipeline.validation()` →
`pipeline.plot(only_validated=...)`, then calls `export_emodels_hoc` and/or
`export_emodels_sonata` with the seeds the user picked. Validation results are
registered as a `TaskResult` and the draft `MEModel` is updated with calibration
results and validation status.

**Reads from:** the entitycore staging project — the optimisation `TaskResult`
entity from notebook 02 and the draft `MEModel` entity.

**Writes to:** `obi-output/03_export_and_validation/grid_scan/0/` — the working
directory plus `final.json`, `figures/`, `export_emodels_hoc/` (one `.hoc` per
validated model), and `export_emodels_sonata/` (with `nodes.h5`).

## Imports

In [ ]:
from pathlib import Path

import obi_one as obi
from entitysdk import Client, ProjectContext
from obi_auth import get_token
from obi_one.core.info import Info
from obi_one.scientific.from_id.task_result_from_id import TaskResultFromID
from obi_one.scientific.tasks.emodel_optimization.task3_export_and_validation.blocks import (
    CurrentscapeConfig,
    ExportAndValidationInitialize,
    ExportAndValidationSettings,
)

## Connect to entitycore staging

The export + validation task downloads the optimisation TaskResult assets
and the draft MEModel from entitycore.

In [ ]:
token = get_token(environment="staging")
project_context = ProjectContext(
    virtual_lab_id=obi.LAB_ID_STAGING_TEST,
    project_id=obi.PROJECT_ID_STAGING_TEST,
)
db_client = Client(
    api_url="https://staging.cell-a.openbraininstitute.org/api/entitycore",
    project_context=project_context,
    token_manager=token,
)
print("Connected to entitycore staging.")

## Build the scan config

The export + validation stage uses entity-based inputs:
- `optimization_task_result`: the `TaskResult` entity from notebook 02
- `memodel`: the draft `MEModel` entity registered by the optimisation stage

Settings control validation (protocols, threshold), plotting (currentscape),
and export (HOC/SONATA, seeds, only_validated, only_best).

In [ ]:
# Replace with real entity IDs from your staging project.
OPTIMIZATION_TASK_RESULT_ID = "00000000-0000-0000-0000-000000000001"
MEMODEL_ID = "00000000-0000-0000-0000-000000000002"

scan_config = obi.EModelExportAndValidationScanConfig(
    info=Info(
        campaign_name="L5PC Export and Validation",
        campaign_description="Validate optimised L5PC models and export to HOC/SONATA.",
    ),
    initialize=ExportAndValidationInitialize(
        optimization_task_result=TaskResultFromID(id_str=OPTIMIZATION_TASK_RESULT_ID),
        memodel=obi.MEModelFromID(id_str=MEMODEL_ID),
        emodel="L5PC",
        species="rat",
        brain_region="SSCX",
        etype="cADpyr",
    ),
    settings=ExportAndValidationSettings(
        validation_protocols="sAHP_220",
        validation_threshold=5.0,
        plot_currentscape=True,
        save_recordings=False,
        only_validated_plots=False,
        export_hoc=True,
        export_sonata=True,
        only_validated=False,
        only_best=False,
        seeds="1",  # must be a subset of the seeds run in stage 02
    ),
    currentscape_config=CurrentscapeConfig(figure_title="L5PC"),
)
print(scan_config.settings.to_dict(scan_config.currentscape_config))

## Run the grid scan

Per-coordinate output goes to `obi-output/03_export_and_validation/grid_scan/<idx>/`.

In [ ]:
grid_scan = obi.GridScanGenerationTask(
    form=scan_config,
    output_root="../../../obi-output/03_export_and_validation/grid_scan",
    coordinate_directory_option="ZERO_INDEX",
)
grid_scan.execute(db_client=db_client)

obi.run_tasks_for_generated_scan(grid_scan, db_client=db_client)

## Inspect the exported model

In [ ]:
coord_root = Path(grid_scan.single_configs[0].coordinate_output_root).resolve()
print("Coordinate output:", coord_root)

for tree in ("export_emodels_hoc", "export_emodels_sonata"):
    print(f"\n{tree}/")
    for p in sorted((coord_root / tree).rglob("*"))[:20]:
        if p.is_file():
            print(" ", p.relative_to(coord_root))

## Inspect validation results

The `final.json` file contains the optimisation results including validation
status, holding current, threshold current, and input resistance.

In [ ]:
import json

final_path = coord_root / "final.json"
if final_path.exists():
    final = json.loads(final_path.read_text())
    print("Final.json keys:", list(final))
    for key, val in final.items():
        if isinstance(val, dict):
            print(f"  {key}: {list(val.keys())}")
        else:
            print(f"  {key}: {val}")
else:
    print("No final.json found.")